In [0]:
dbutils.widgets.removeAll()

In [0]:
%run ./helper

In [0]:
dbutils.widgets.dropdown("source_catalog", catalog_list[0], catalog_list)
dbutils.widgets.dropdown("source_schema", schema_list[0], schema_list)
dbutils.widgets.dropdown("model_result_table", model_result_table_list[0], model_result_table_list)
dbutils.widgets.dropdown("instruct_model", instruct_model_list[0], instruct_model_list)
catalog=dbutils.widgets.get("source_catalog")
schema=dbutils.widgets.get("source_schema")
model_result_table=dbutils.widgets.get("model_result_table")
instruct_model=dbutils.widgets.get("instruct_model")
df = spark.read.table(f"{catalog}.{schema}.{model_result_table}")
table_name = f"{catalog}.{schema}.{model_result_table}"
print(f"table_name:{table_name}, instruct_model:{instruct_model}" )


In [0]:
ai_grade_str="""ai_query("{}",
concat(
concat("give a grade from 0-5 for the cluster of the following sentences, if they are related to each other, no prefixes only grade and one sentence explanation in english: ",  array_join(Representative_Docs, ', ')
),
concat("reprsented by the following words: ", array_join(Representation, ', '))
) 
)"""

In [0]:
ai_grade_str=ai_grade_str.format(instruct_model)

In [0]:
sql_str=f"""select *, {ai_grade_str} as ai_grade from {table_name}"""

In [0]:
result_df=spark.sql(sql_str)

In [0]:
result_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable(f"{catalog}.{schema}.topic_info_local")
#amit.default.topic_info_local

In [0]:
spark.table(f"{catalog}.{schema}.topic_info_local").display()